# 📊 Sales Analysis - AAL (Australia Apparel Ltd)

Fourth Quarter Sales Analysis (2020)

---

## Project Statement
AAL is analyzing its Q4 sales data across Australian states to:
1. Identify high revenue states
2. Improve low-performing regions
3. Enable data-driven expansion decisions


In [49]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import FancyBboxPatch
import plotly.express as px

sns.set(style="whitegrid")



In [50]:
# Load dataset into a pandas dataframe
df = pd.read_csv('./Data/AusApparalSales4thQrt2020.csv')
df.head()

,Date,Time,State,Group,Unit,Sales
0,1-Oct-2020,Morning,WA,Kids,8,20000
1,1-Oct-2020,Morning,WA,Men,8,20000
2,1-Oct-2020,Morning,WA,Women,4,10000
3,1-Oct-2020,Morning,WA,Seniors,15,37500
4,1-Oct-2020,Afternoon,WA,Kids,3,7500


In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7560 entries, 0 to 7559
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    7560 non-null   object
 1   Time    7559 non-null   object
 2   State   7560 non-null   object
 3   Group   7560 non-null   object
 4   Unit    7560 non-null   int64 
 5   Sales   7560 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 354.5+ KB


## 1. Data Wrangling

In [52]:
# Cleaning the data and column has extra spaces. Remove the extra spaces in the columns
df['Time'] = df['Time'].str.strip()
df['State'] = df['State'].str.strip()
df['Group'] = df['Group'].str.strip()

In [53]:
#
df['Date'] = pd.to_datetime(df['Date'])

In [54]:
# Check missing values
df.isna().sum()

Date     0
Time     1
State    0
Group    0
Unit     0
Sales    0
dtype: int64

### Handling Missing Data
- Drop rows if minimal missing values
- Fill using mean/median if necessary


In [55]:
# Example: fill missing numeric values
df.fillna(df.mean(numeric_only=True), inplace=True)

### Normalization

In [56]:
# Normalize Sales column
df['Sales_Normalized'] = (df['Sales'] - df['Sales'].min()) / (df['Sales'].max() - df['Sales'].min())
df.head()

,Date,Time,State,Group,Unit,Sales,Sales_Normalized
0,2020-10-01,Morning,WA,Kids,8,20000,0.095238
1,2020-10-01,Morning,WA,Men,8,20000,0.095238
2,2020-10-01,Morning,WA,Women,4,10000,0.031746
3,2020-10-01,Morning,WA,Seniors,15,37500,0.206349
4,2020-10-01,Afternoon,WA,Kids,3,7500,0.015873


### GroupBy Insight

In [57]:
# Group by State
state_group = df.groupby('State')['Sales'].sum()
state_group

State
NSW     74970000
NT      22580000
QLD     33417500
SA      58857500
TAS     22760000
VIC    105565000
WA      22152500
Name: Sales, dtype: int64

## 📉 2. Data Analysis

In [58]:
# Descriptive statistics
df[['Sales','Unit']].describe()

,Sales,Unit
count,7560.000000,7560.000000
mean,45013.558201,18.005423
std,32253.506944,12.901403
min,5000.000000,2.000000
25%,20000.000000,8.000000
50%,35000.000000,14.000000
75%,65000.000000,26.000000
max,162500.000000,65.000000


#### 2.a: Perform descriptive statistical analysis on the data in the Sales and Unit columns. 
 - Utilize techniques such as `mean`, `median`, `mode`, and `standard deviation` for this analysis. 

In [59]:
# Mean, Median, Mode, Std
stats_sales = {
    'Mean': df['Sales'].mean(),
    'Median': df['Sales'].median(),
    'Mode': df['Sales'].mode()[0],
    'Std': df['Sales'].std()
}
stats_unit = {
    'Mean': df['Unit'].mean(),
    'Median': df['Unit'].median(),
    'Mode': df['Unit'].mode()[0],
    'Std': df['Unit'].std()
}
print("💵 Sales Statistics:")
print(f"  Mean: { round(stats_sales['Mean'], 2)}")
print(f"  Median: { round(stats_sales['Median'], 2)}")
print(f"  Mode: {stats_sales['Mode']}")
print(f"  Std: {round(stats_sales['Std'], 2)}")
print("")
print("📦 Unit Statistics:")
print(f"  Mean: { round(stats_unit['Mean'], 2)}")
print(f"  Median: {round(stats_unit['Median'], 2)}")
print(f"  Mode: {stats_unit['Mode']}")
print(f"  Std: {round(stats_unit['Std'], 2)}")

💵 Sales Statistics:
  Mean: 45013.56
  Median: 35000.0
  Mode: 22500
  Std: 32253.51

📦 Unit Statistics:
  Mean: 18.01
  Median: 14.0
  Mode: 9
  Std: 12.9


#### 2.b : Identify the group with the highest sales and the group with the lowest sales based on the data provided. 
#### 2.c : Identify the group with the highest and lowest sales based on the data provided.

In [60]:
# Highest and lowest sales states
state_group = df.groupby('State')['Sales'].sum()
highest_state = state_group.idxmax()
lowest_state = state_group.idxmin()

print(f"📊 Highest Sales State: {highest_state} with ${state_group[highest_state]:,.2f}")
print(f"📉 Lowest Sales State: {lowest_state} with ${state_group[lowest_state]:,.2f}")

📊 Highest Sales State: VIC with $105,565,000.00
📉 Lowest Sales State: WA with $22,152,500.00


#### 2.d. Generate weekly, monthly, and quarterly reports to document and present the results of the analysis conducted.

In [61]:
# Setting Date as index for time series analysis
df.set_index('Date', inplace=True)

In [62]:
# Populate the weekly sales matrix

weekly_df = df.resample('W')['Sales'].sum().apply(lambda x: f"${x:,.2f}")
weekly_df.name = 'Weekly Sales'
weekly_df= weekly_df.to_frame(name='Weekly Sales($)')

print(f"📊 Weekly Sales Data:")
print(weekly_df.head())

# Calculate monthly sales
monthly_df = df.resample('ME')['Sales'].sum().apply(lambda x: f"${x:,.2f}")
monthly_df.name = 'Monthly Sales'
monthly_df = monthly_df.to_frame(name='Monthly Sales($)')

print(f"\n📊 Monthly Sales Data:")
print(monthly_df.head())


# Quarterly sales
quarterly_df = df.resample('QE')['Sales'].sum().apply(lambda x: f"${x:,.2f}")
quarterly_df.name = 'Quarterly Sales'
quarterly_df = quarterly_df.to_frame(name='Quarterly Sales($)')

print(f"\n📊 Quarterly Sales Data:")
print(quarterly_df.head()) 



📊 Weekly Sales Data:
           Weekly Sales($)
Date                      
2020-10-04  $15,045,000.00
2020-10-11  $27,002,500.00
2020-10-18  $26,640,000.00
2020-10-25  $26,815,000.00
2020-11-01  $21,807,500.00

📊 Monthly Sales Data:
           Monthly Sales($)
Date                       
2020-10-31  $114,290,000.00
2020-11-30   $90,682,500.00
2020-12-31  $135,330,000.00

📊 Quarterly Sales Data:
           Quarterly Sales($)
Date                         
2020-12-31    $340,302,500.00


## 3. Data Visualization

#### 📊 State-wise sales analysis 

In [63]:
import plotly.express as px

# State-wise sales
state_group_df = df.groupby('State')['Sales'].sum().reset_index()

# Create donut chart
fig = px.pie(
    state_group_df,
    names='State',
    values='Sales',
    title="Sales ($) by State",
    hole=0.4,  # makes it a donut
    color_discrete_sequence=px.colors.sequential.Viridis
)

# Add $M labels inside
fig.update_traces(
    text=state_group_df['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),
    textposition='auto',
    textfont=dict(size=10, color='white')
)

# Clean layout
fig.update_layout(
    template="plotly_dark",
    width = 500,   # pixels
    height = 500,   # pixels
    legend=dict(
        orientation="h",
        y=-0.2,
        x=0.5,
        xanchor="center"
    )
)

fig.show()

#### 📊 State-wise sales analysis for different demographic groups (kids, women, men, and seniors). 

In [64]:


# Prepare data (same logic as yours)
state_group_df = df.groupby(['State', 'Group'])['Sales'].sum().reset_index()

# Create stacked bar chart
fig = px.bar(
    state_group_df,
    x='State',
    y='Sales',
    color='Group',
    title='Sales ($) by State and Group',
    text=state_group_df['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),
    color_discrete_sequence=px.colors.sequential.Aggrnyl
)

# Format Y-axis in $ Millions
fig.update_layout(
    yaxis_title="Sales ($ Millions)",
    xaxis_title="State",
    yaxis=dict(tickformat=".2s")  # auto scales (K, M, B)
)

# Put labels inside bars
fig.update_traces(textposition='auto')

# Clean layout
fig.update_layout(
    template="plotly_dark",
    legend_title="Group"
)

fig.show()

In [76]:
# Example: aggregate by Date
daily_sales = df.groupby('Date')['Sales'].sum().reset_index()
#daily_sales = df.resample('W')['Sales'].sum().to_frame(name='Sales').reset_index()

# Create line chart
fig = px.line(
    daily_sales,
    x='Date',
    y='Sales',
    title='Sales Trend Over Time'
)

# Format Y-axis in $ Millions
fig.update_yaxes(
    tickprefix="$",
    tickformat=".2s"   # auto K, M, B
)

# Add data labels (optional)
fig.update_traces(
    text=daily_sales['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),
    textposition="top center",
    mode='lines+markers',       # lines + markers
    line=dict(color='cyan', width=2),           # line color
    marker=dict(size=10, color='cyan', symbol='circle'),  # same as line
)

# Clean layout
fig.update_layout(
    template="plotly_dark",
    width=900,
    height=500
)

fig.show()

In [82]:

# Example: aggregate sales by State
# state_group_df = df.groupby('State')['Sales'].sum().reset_index()
monthly_sales = df.resample('ME')['Sales'].sum().to_frame(name='Sales').reset_index()

# Create bar chart
fig = px.bar(
    monthly_sales,
    x='Date',
    y='Sales',
    title='Monthly Sales',
    text=monthly_sales['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),  # add labels
    color_discrete_sequence=px.colors.sequential.Viridis
)

# Show labels on top of bars
fig.update_traces(textposition='outside')

# Format X-axis to Mon-YY
fig.update_xaxes(
    tickformat="%b-%y",  
    dtick="M1"            # tick every 1 month
)

# Format Y-axis in $ Millions
fig.update_yaxes(
    tickprefix="$",
    tickformat=".2s"  
)

# Layout
fig.update_layout(
    template='plotly_dark',
    width=800,
    height=500,
    showlegend=False  # hide legend if color matches x-axis
)

fig.show()

In [ ]:
monthly_sales = df.resample('QE')['Sales'].sum().to_frame(name='Sales').reset_index()

# Create bar chart
fig = px.bar(
    monthly_sales,
    x='Date',
    y='Sales',
    title='Monthly Sales',
    text=monthly_sales['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),  # add labels
    color_discrete_sequence=px.colors.sequential.Viridis
)

# Show labels on top of bars
fig.update_traces(textposition='outside')

# Format X-axis to Mon-YY
fig.update_xaxes(
    tickformat="%b-%y",  
    dtick="M1"           
)

# Format Y-axis in $ Millions
fig.update_yaxes(
    tickprefix="$",
    tickformat=".2s"  
)

# Layout
fig.update_layout(
    template='plotly_dark',
    width=800,
    height=500,
    showlegend=False  # hide legend if color matches x-axis
)

fig.show()

In [88]:
import plotly.express as px

# Time-wise sales
time_group_df = df.groupby('Time')['Sales'].sum().reset_index()

# Create donut chart
fig = px.pie(
    time_group_df,
    names='Time',
    values='Sales',
    title="Sales ($) by Time",
    hole=0.4,  # makes it a donut
    color_discrete_sequence=px.colors.sequential.Viridis
)

# Add $M labels inside
fig.update_traces(
    text=time_group_df['Sales'].apply(lambda x: f"${x/1e6:.2f} M"),
    textposition='auto',
    textfont=dict(size=10, color='white')
)

# Clean layout
fig.update_layout(
    template="plotly_dark",
    width = 500,   # pixels
    height = 500,   # pixels
    legend=dict(
        orientation="h",
        y=-0.2,
        x=0.5,
        xanchor="center"
    )
)

fig.show()

## 4. Recommendations
- Focus marketing in low-performing states
- Use peak time insights for promotions
- Expand high-performing regions
- Seaborn used for statistical clarity and simplicity
